In [ ]:
import matplotlib as mpl
mpl.rcParams['axes.formatter.use_mathtext'] = False

%matplotlib widget
%load_ext autoreload
%autoreload 2
from IPython.display import display, HTML
css = """
<style>
    /*overwrite hard coded write background by vscode for ipywidges */
    .cell-output-ipywidget-background {
    background-color: transparent !important;
    }

    /*set widget foreground text and color of interactive widget to vs dark theme color */
    :root {
        --jp-widgets-color: var(--vscode-editor-foreground);
        --jp-widgets-font-size: var(--vscode-editor-font-size);
    }
</style>
"""
display(HTML(css))

import matplotlib.pyplot as plt
import numpy as np
# plt.style.use("dark_background")


try: # if using the new qcodes based instrumet server
    from instrumentserver.client import ClientStation  
    instrument_station = ClientStation(host='localhost', timeout=600, init_instruments = None)
except Exception as e: # make a fall back dummy object
    instrument_station =  type('Dummy', (), {})()
    instrument_station.save_parameters = lambda *args:  None


# a simple python file that stores global settings
# this currently uses a very simple relative path, remember to change this if you move the modules around
from system_info import DATA_DIR, BOARD_IP, YAML_PATH



# Alice BS Amp Freq Sweep

In [ ]:
from acadia_qmsmt.runtimes.bs_amp_freq_sweep_pulseSSB import BSAmpFreqSweepPulseSSBRuntime

bs_stimulus = "qb_alice_bs_stimulus"
bs_detunes = np.linspace(-1e6, 1e6, 41)
bs_amplitudes = np.linspace(2e-2, 3e-1, 81)
bs_pulse = "swap_selective" # define a separate swap pulse that can be much longer as you search for precise freq

iterations = 500
run_delay = 1000e3
capture_window_name = "matched"

use_cool = False
cool_qm_rounds = 2

rt = BSAmpFreqSweepPulseSSBRuntime(qubit_stimulus="qb_stimulus", readout_stimulus="ro_stimulus", readout_capture="ro_capture", bs_stimulus=bs_stimulus,
                        bs_amplitudes=bs_amplitudes, bs_detunes=bs_detunes, bs_pulse_name=bs_pulse,
                        capture_window_name=capture_window_name,
                        plot=True, iterations=iterations, run_delay=run_delay,
                        use_cool=use_cool, cool_qm_rounds=cool_qm_rounds, yaml_path=YAML_PATH)
rt.deploy(BOARD_IP, local_directory=DATA_DIR + f"BS_AmpFreqSweep/{'_'.join(bs_stimulus.split('_')[:2])}/%y%m%d/%H%M%S")
rt.display()
_ = instrument_station.save_parameters(rt.local_directory + "/inst_params.json")

# Alice Chevron

In [ ]:
from acadia_qmsmt.runtimes.bs_chevron import BSChevronRuntime


bs_stimulus = "qb_alice_bs_stimulus"
flat_length_list = np.linspace(0, 4e-6, 101) + 10e-9
bs_frequencies = np.linspace(-1.5e6, 1.5e6, 81) + 2.6848e9
bs_amp = 1e-1 # None use yaml value

iterations = 500
run_delay = 1000e3
capture_window_name = "matched"
bs_pulse = "swap_stretchable"

cool_qm_rounds = 0

rt = BSChevronRuntime(qubit_stimulus="qb_stimulus", readout_stimulus="ro_stimulus", readout_capture="ro_capture", bs_stimulus=bs_stimulus,
                        flat_length_list=flat_length_list, bs_frequencies=bs_frequencies,
                        capture_window_name=capture_window_name,
                        plot=True, iterations=iterations, run_delay=run_delay,
                        bs_amp=bs_amp, bs_pulse_name=bs_pulse,
                        cool_qm_rounds=cool_qm_rounds, yaml_path=YAML_PATH)
rt.deploy(BOARD_IP, local_directory=DATA_DIR + f"BS_Chevron/{'_'.join(bs_stimulus.split('_')[:2])}/%y%m%d/%H%M%S")
rt.display()
_ = instrument_station.save_parameters(rt.local_directory + "/inst_params.json")

# Alice T1

In [ ]:
from acadia_qmsmt.runtimes.QM_T1_viaSWAP import QMT1Runtime

bs_stimulus = "qb_alice_bs_stimulus"
delay_times = np.linspace(0, 800e-6, 51) + 50e-9

iterations = 5000
run_delay = 1000e3
capture_window_name = "matched"
bs_pulse = "swap"
qubit_pulse = "R_x_180"

cool_qm_rounds = 0

rt = QMT1Runtime(qubit_stimulus="qb_stimulus", readout_stimulus="ro_stimulus", readout_capture="ro_capture", bs_stimulus=bs_stimulus,
                        capture_window_name=capture_window_name, plot=True, iterations=iterations, run_delay=run_delay,
                        delay_times=delay_times, qubit_pulse_name=qubit_pulse, bs_pulse_name=bs_pulse, cool_qm_rounds=cool_qm_rounds, yaml_path=YAML_PATH)


rt.deploy(BOARD_IP, local_directory=DATA_DIR + f"QM_T1_SWAP/{'_'.join(bs_stimulus.split('_')[:2])}/%y%m%d/%H%M%S")
rt.display()
_ = instrument_station.save_parameters(rt.local_directory + "/inst_params.json")

# Alice T2

In [ ]:
from acadia_qmsmt.runtimes.QM_T2_viaSWAP import QMT2Runtime

bs_stimulus = "qb_alice_bs_stimulus"
delay_times = np.linspace(0, 500e-6, 51) + 10e-9

iterations = 50000
run_delay = 800e3
capture_window_name = "matched"
bs_pulse = "swap"
qubit_pi_pulse = "R_x_180"

soft_detune = 0.02e6
do_echo = False

cool_qm_rounds = 0

rt = QMT2Runtime(qubit_stimulus="qb_stimulus", readout_stimulus="ro_stimulus", readout_capture="ro_capture", bs_stimulus=bs_stimulus,
                        capture_window_name=capture_window_name, plot=True, iterations=iterations, run_delay=run_delay,
                        soft_detune=soft_detune, do_echo=do_echo, delay_times=delay_times, qubit_pi_pulse_name=qubit_pi_pulse, 
                        bs_pulse_name=bs_pulse, yaml_path=YAML_PATH)


rt.deploy(BOARD_IP, local_directory=DATA_DIR + f"QM_T2_SWAP/{'_'.join(bs_stimulus.split('_')[:2])}/%y%m%d/%H%M%S")
rt.display()
_ = instrument_station.save_parameters(rt.local_directory + "/inst_params.json")

#  Bob BS Amp freq sweep

In [ ]:
from acadia_qmsmt.runtimes.bs_amp_freq_sweep_pulseSSB import BSAmpFreqSweepPulseSSBRuntime

bs_stimulus = "qb_bob_bs_stimulus"
bs_detunes = np.linspace(-0.5e6, 0.5e6, 101)
bs_amplitudes = np.linspace(0.02, 0.1, 81)
bs_pulse = "swap_selective" 

iterations = 500
run_delay = 1000e3
capture_window_name = "matched"

use_cool = False
cool_qm_rounds = 0

rt = BSAmpFreqSweepPulseSSBRuntime(qubit_stimulus="qb_stimulus", readout_stimulus="ro_stimulus", readout_capture="ro_capture", bs_stimulus=bs_stimulus,
                        bs_amplitudes=bs_amplitudes, bs_detunes=bs_detunes, bs_pulse_name=bs_pulse,
                        capture_window_name=capture_window_name,
                        plot=True, iterations=iterations, run_delay=run_delay,
                        use_cool=use_cool, cool_qm_rounds=cool_qm_rounds, yaml_path=YAML_PATH)
rt.deploy(BOARD_IP, local_directory=DATA_DIR + "BS_AmpFreqSweep/%y%m%d/%H%M%S")
rt.display()
_ = instrument_station.save_parameters(rt.local_directory + "/inst_params.json")

Output()

[2025-07-18 15:48:08,131] WARNING at event_loop (runtime.py, 646): Unable to connect to target DataManager
[2025-07-18 15:48:17,479] ERROR at event_loop (runtime.py, 658): Exception synchronizing: Traceback (most recent call last):
  File "/home/npkhedkar/acadia_env/lib/python3.10/site-packages/acadia/runtime.py", line 655, in event_loop
    self.data.sync(timeout_ms=self._datamanager_sync_timeout_ms)
TimeoutError: Timeout occurred waiting for line

[2025-07-18 15:48:27,555] ERROR at event_loop (runtime.py, 658): Exception synchronizing: Traceback (most recent call last):
  File "/home/npkhedkar/acadia_env/lib/python3.10/site-packages/acadia/runtime.py", line 655, in event_loop
    self.data.sync(timeout_ms=self._datamanager_sync_timeout_ms)
TimeoutError: Timeout occurred waiting for line

[2025-07-18 15:48:37,634] ERROR at event_loop (runtime.py, 658): Exception synchronizing: Traceback (most recent call last):
  File "/home/npkhedkar/acadia_env/lib/python3.10/site-packages/acadia/run

# BS Chevron

In [ ]:
from acadia_qmsmt.runtimes.bs_chevron import BSChevronRuntime


bs_stimulus = "qb_bob_bs_stimulus"
flat_length_list = np.linspace(0, 5e-6, 101) + 10e-9
bs_frequencies = np.linspace(-5e6, 5e6, 81) + 1.71e9 # 1.7507e9
bs_amp = 8e-2 # None use yaml value

iterations = 500
run_delay = 1000e3
capture_window_name = "matched"
bs_pulse = "swap_stretchable"

cool_qm_rounds = 0

rt = BSChevronRuntime(qubit_stimulus="qb_stimulus", readout_stimulus="ro_stimulus", readout_capture="ro_capture", bs_stimulus=bs_stimulus,
                        flat_length_list=flat_length_list, bs_frequencies=bs_frequencies,
                        capture_window_name=capture_window_name,
                        plot=True, iterations=iterations, run_delay=run_delay,
                        bs_amp=bs_amp, bs_pulse_name=bs_pulse,
                        cool_qm_rounds=cool_qm_rounds, yaml_path=YAML_PATH)
rt.deploy(BOARD_IP, local_directory=DATA_DIR + f"BS_Chevron/{'_'.join(bs_stimulus.split('_')[:2])}/%y%m%d/%H%M%S")
rt.display()
_ = instrument_station.save_parameters(rt.local_directory + "/inst_params.json")

Output()

[2025-07-20 23:49:54,118] WARNING at event_loop (runtime.py, 646): Unable to connect to target DataManager


Can't save figure prep msmts generated by plot_prep_msmts: 'Axes' object is not subscriptable


# Bob T1

In [35]:
from acadia_qmsmt.runtimes.QM_T1_viaSWAP import QMT1Runtime

bs_stimulus = "qb_bob_bs_stimulus"
delay_times = np.linspace(0, 250e-6, 51) + 60e-9

iterations = 5000
run_delay = 2000e3
capture_window_name = "matched"
bs_pulse = "swap"
qubit_pulse = "R_x_180"

cool_qm_rounds = 0

rt = QMT1Runtime(qubit_stimulus="qb_stimulus", readout_stimulus="ro_stimulus", readout_capture="ro_capture", bs_stimulus=bs_stimulus,
                        capture_window_name=capture_window_name, plot=True, iterations=iterations, run_delay=run_delay,
                        delay_times=delay_times, qubit_pulse_name=qubit_pulse, bs_pulse_name=bs_pulse, cool_qm_rounds=cool_qm_rounds, yaml_path=YAML_PATH)


rt.deploy(BOARD_IP, local_directory=DATA_DIR + f"QM_T1_SWAP/{'_'.join(bs_stimulus.split('_')[:2])}/%y%m%d/%H%M%S")
rt.display()
_ = instrument_station.save_parameters(rt.local_directory + "/inst_params.json")

Output()

[2025-07-20 22:47:35,068] WARNING at event_loop (runtime.py, 646): Unable to connect to target DataManager


# Bob T2

In [ ]:
from acadia_qmsmt.runtimes.QM_T2_viaSWAP import QMT2Runtime

bs_stimulus = "qb_bob_bs_stimulus"
delay_times = np.linspace(0, 150e-6, 51) + 50e-9

iterations = 2000
run_delay = 2000e3
capture_window_name = "matched"
bs_pulse = "swap"
qubit_pi_pulse = "R_x_180"

soft_detune = 0.03e6
do_echo = False 

cool_qm_rounds = 0

rt = QMT2Runtime(qubit_stimulus="qb_stimulus", readout_stimulus="ro_stimulus", readout_capture="ro_capture", bs_stimulus=bs_stimulus,
                        capture_window_name=capture_window_name, plot=True, iterations=iterations, run_delay=run_delay,
                        soft_detune=soft_detune, do_echo=do_echo, delay_times=delay_times, qubit_pi_pulse_name=qubit_pi_pulse, 
                        bs_pulse_name=bs_pulse, yaml_path=YAML_PATH)


rt.deploy(BOARD_IP, local_directory=DATA_DIR + f"QM_T2_SWAP/{'_'.join(bs_stimulus.split('_')[:2])}/%y%m%d/%H%M%S")
rt.display()
_ = instrument_station.save_parameters(rt.local_directory + "/inst_params.json")

Output()

[2025-07-20 22:41:50,595] WARNING at event_loop (runtime.py, 646): Unable to connect to target DataManager


# Optional:  Coupler BS Tuneup

# Coupler Chevron

In [ ]:
from acadia_qmsmt.runtimes.bs_chevron import BSChevronRuntime


bs_stimulus = "qb_linc_bs_stimulus" # this will be different depending on coupler type
flat_length_list = np.linspace(0, 0.2e-6, 41) + 10e-9
bs_frequencies = np.linspace(-15e6, 15e6, 81) + 1.446e9
bs_amp = 1e-1 # None use yaml value

iterations = 500
run_delay = 20e3
capture_window_name = "matched"
bs_pulse = "swap_stretchable"

cool_qm_rounds = 0

rt = BSChevronRuntime(qubit_stimulus="qb_stimulus", readout_stimulus="ro_stimulus", readout_capture="ro_capture", bs_stimulus=bs_stimulus,
                        flat_length_list=flat_length_list, bs_frequencies=bs_frequencies,
                        capture_window_name=capture_window_name,
                        plot=True, iterations=iterations, run_delay=run_delay,
                        bs_amp=bs_amp, bs_pulse_name=bs_pulse,
                        cool_qm_rounds=cool_qm_rounds, yaml_path=YAML_PATH)
rt.deploy(BOARD_IP, local_directory=DATA_DIR + f"BS_Chevron/{'_'.join(bs_stimulus.split('_')[:2])}/%y%m%d/%H%M%S")
rt.display()
_ = instrument_station.save_parameters(rt.local_directory + "/inst_params.json")

Output()

[2025-07-21 14:39:07,233] WARNING at event_loop (runtime.py, 646): Unable to connect to target DataManager


# Coupler T1

In [61]:
from acadia_qmsmt.runtimes.QM_T1_viaSWAP import QMT1Runtime

bs_stimulus = "qb_linc_bs_stimulus"
delay_times = np.linspace(0, 80e-6, 51) + 20e-9

iterations = 5000
run_delay = 80e3
capture_window_name = "matched"
bs_pulse = "swap"
qubit_pulse = "R_x_180"

cool_qm_rounds = 0

rt = QMT1Runtime(qubit_stimulus="qb_stimulus", readout_stimulus="ro_stimulus", readout_capture="ro_capture", bs_stimulus=bs_stimulus,
                        capture_window_name=capture_window_name, plot=True, iterations=iterations, run_delay=run_delay,
                        delay_times=delay_times, qubit_pulse_name=qubit_pulse, bs_pulse_name=bs_pulse, cool_qm_rounds=cool_qm_rounds, yaml_path=YAML_PATH)


rt.deploy(BOARD_IP, local_directory=DATA_DIR + f"QM_T1_SWAP/{'_'.join(bs_stimulus.split('_')[:2])}/%y%m%d/%H%M%S")
rt.display()
_ = instrument_station.save_parameters(rt.local_directory + "/inst_params.json")

Output()

[2025-07-21 14:22:26,295] WARNING at event_loop (runtime.py, 646): Unable to connect to target DataManager


# Coupler T2

In [92]:
from acadia_qmsmt.runtimes.QM_T2_viaSWAP import QMT2Runtime

bs_stimulus = "qb_linc_bs_stimulus"
delay_times = np.linspace(0, 100e-9, 11) + 10e-9

iterations = 5000
run_delay = 80e3
capture_window_name = "matched"
bs_pulse = "swap"
qubit_pi_pulse = "R_x_180"

soft_detune = 40e6
do_echo = False

cool_qm_rounds = 0

rt = QMT2Runtime(qubit_stimulus="qb_stimulus", readout_stimulus="ro_stimulus", readout_capture="ro_capture", bs_stimulus=bs_stimulus,
                        capture_window_name=capture_window_name, plot=True, iterations=iterations, run_delay=run_delay,
                        soft_detune=soft_detune, do_echo=do_echo, delay_times=delay_times, qubit_pi_pulse_name=qubit_pi_pulse, 
                        bs_pulse_name=bs_pulse, yaml_path=YAML_PATH)


rt.deploy(BOARD_IP, local_directory=DATA_DIR + f"QM_T2_SWAP/{'_'.join(bs_stimulus.split('_')[:2])}/%y%m%d/%H%M%S")
rt.display()
_ = instrument_station.save_parameters(rt.local_directory + "/inst_params.json")

Output()

[2025-07-21 14:50:50,204] WARNING at event_loop (runtime.py, 646): Unable to connect to target DataManager
